# 01b Efecto de la simplificación del grafo (Colab)

**Proyecto:** accesibilidad peatonal a equipamiento urbano – Miraflores + San Juan de Miraflores (Hito 1)

El enunciado pide usar `simplify=True` y **explicar su efecto sobre el conteo de nodos**. OpenStreetMap modela cada calle con muchos nodos de geometría (puntos intermedios de una curva) que no son intersecciones; la simplificación los elimina y deja solo intersecciones y finales de vía, fusionando los tramos en una arista con su geometría.

Este notebook descarga **una sola vez** el grafo **sin simplificar** de la misma zona que en `01` y lo simplifica con `ox.simplify_graph`, de modo que ambas versiones salen de los mismos datos y la comparación es limpia. Requiere acceso a Overpass (por eso se ejecuta en Colab). Al terminar, copiar `simplify_comparacion.json` a `data/raw/` del repositorio.

## 1. Configuración y zona de descarga

In [1]:
import sys, json, time, platform, datetime, subprocess, importlib.util
from pathlib import Path

if importlib.util.find_spec("osmnx") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "osmnx"])

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox

try:
    sys.path.insert(0, str(Path.cwd().parent))
    from src import config as cfg
    DATA_RAW = cfg.DATA_RAW
except Exception:
    class cfg:   # Colab sin el repositorio: mismos valores que src/config.py
        SEED = 42
        PLACES = ["Miraflores, Lima, Peru", "San Juan de Miraflores, Lima, Peru"]
        DISTRICT_LABELS = ["Miraflores", "San Juan de Miraflores"]
        BUFFER_M = 300
        NETWORK_TYPE = "walk"
        CRS_METRIC = "EPSG:32718"
    DATA_RAW = Path("data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

np.random.seed(cfg.SEED)
ox.settings.timeout = 300
ox.settings.requests_timeout = 120
ox.settings.use_cache = True
ox.settings.log_console = False

limites = ox.geocode_to_gdf(cfg.PLACES)
limites["distrito"] = cfg.DISTRICT_LABELS
limites_utm = limites.to_crs(cfg.CRS_METRIC)
poligono = gpd.GeoSeries([limites_utm.geometry.union_all().buffer(cfg.BUFFER_M)], crs=cfg.CRS_METRIC).to_crs(4326).iloc[0]
print("osmnx", ox.__version__, "| polígono:", poligono.geom_type, "| buffer", cfg.BUFFER_M, "m")

osmnx 2.1.1 | polígono: MultiPolygon | buffer 300 m


## 2. Descarga sin simplificar y simplificación

In [2]:
momento = datetime.datetime.now().astimezone()
for intento in range(1, 7):
    try:
        G_ns = ox.graph_from_polygon(poligono, network_type=cfg.NETWORK_TYPE, simplify=False, retain_all=True)
        break
    except Exception as e:
        print(f"Intento {intento} fallido: {type(e).__name__}")
        if intento == 6:
            raise
        time.sleep(10)

G_s = ox.simplify_graph(G_ns)
print(f"Sin simplificar: {G_ns.number_of_nodes():,} nodos, {G_ns.number_of_edges():,} aristas")
print(f"Simplificado:    {G_s.number_of_nodes():,} nodos, {G_s.number_of_edges():,} aristas")

Sin simplificar: 42,545 nodos, 102,158 aristas
Simplificado:    17,787 nodos, 52,642 aristas


## 3. Comparación

Se cuentan nodos y aristas (dirigidas), la longitud media de arista y los nodos de geometría (nodos por los que pasan solo dos tramos de calle, `street_count = 2`, que son los que la simplificación elimina). También se cuenta por distrito (un nodo pertenece al distrito cuyo polígono lo contiene).

In [3]:
def por_distrito(G):
    n = ox.graph_to_gdfs(G, edges=False).to_crs(cfg.CRS_METRIC)
    j = gpd.sjoin(n[["geometry"]], limites_utm[["distrito", "geometry"]], how="left", predicate="within")
    j = j[~j.index.duplicated(keep="first")]["distrito"].fillna("Fuera (buffer)")
    return j.value_counts().to_dict()

def resumen(G):
    aristas = ox.graph_to_gdfs(G, nodes=False)
    sc = pd.Series(nx.get_node_attributes(G, "street_count"))
    return {
        "nodos": G.number_of_nodes(),
        "aristas": G.number_of_edges(),
        "longitud media de arista (m)": float(aristas["length"].mean()),
        "longitud total (km, aristas dirigidas / 2)": float(aristas["length"].sum() / 2000),
        "nodos de geometría (street_count = 2)": int((sc == 2).sum()),
        "nodos de intersección (street_count >= 3)": int((sc >= 3).sum()),
        "calles sin salida (street_count = 1)": int((sc == 1).sum()),
        "componentes (débiles)": nx.number_weakly_connected_components(G),
    }

res_ns, res_s = resumen(G_ns), resumen(G_s)
dist_ns, dist_s = por_distrito(G_ns), por_distrito(G_s)
tabla = pd.DataFrame({"sin simplificar": res_ns, "simplificado": res_s})
tabla["cambio"] = tabla["simplificado"] - tabla["sin simplificar"]
tabla["cambio %"] = (tabla["cambio"] / tabla["sin simplificar"] * 100).round(1)
display(tabla.round(2))

tabla_d = pd.DataFrame({"sin simplificar": dist_ns, "simplificado": dist_s}).fillna(0).astype(int)
tabla_d["nodos eliminados"] = tabla_d["sin simplificar"] - tabla_d["simplificado"]
tabla_d["% eliminado"] = (tabla_d["nodos eliminados"] / tabla_d["sin simplificar"] * 100).round(1)
display(tabla_d)

,sin simplificar,simplificado,cambio,cambio %
nodos,42545.00,17787.00,-24758.00,-58.2
aristas,102158.00,52642.00,-49516.00,-48.5
longitud media de arista (m),24.55,47.50,22.95,93.5
"longitud total (km, aristas dirigidas / 2)",1254.14,1250.25,-3.90,-0.3
nodos de geometría (street_count = 2),24711.00,202.00,-24509.00,-99.2
nodos de intersección (street_count >= 3),15729.00,15480.00,-249.00,-1.6
calles sin salida (street_count = 1),2105.00,2105.00,0.00,0.0
componentes (débiles),138.00,123.00,-15.00,-10.9


,sin simplificar,simplificado,nodos eliminados,% eliminado
San Juan de Miraflores,19617,9367,10250,52.3
Fuera (buffer),12069,4492,7577,62.8
Miraflores,10859,3928,6931,63.8


## 4. Registro y descarga del archivo

Se guarda un JSON con la fecha, los parámetros, las versiones y ambas tablas. Los grafos no se guardan (el simplificado ya está en `data/raw/walk_graph_raw.graphml`; el no simplificado es solo para esta comparación).

In [4]:
log = {
    "fecha_descarga": momento.isoformat(timespec="seconds"),
    "consulta": {"funcion": "ox.graph_from_polygon(simplify=False) + ox.simplify_graph", "lugares": cfg.PLACES,
                 "network_type": cfg.NETWORK_TYPE, "retain_all": True, "buffer_m": cfg.BUFFER_M},
    "versiones": {"python": platform.python_version(), "osmnx": ox.__version__, "networkx": nx.__version__,
                  "geopandas": gpd.__version__, "pandas": pd.__version__, "numpy": np.__version__},
    "sin_simplificar": res_ns,
    "simplificado": res_s,
    "nodos_por_distrito": {"sin_simplificar": dist_ns, "simplificado": dist_s},
}
ruta = DATA_RAW / "simplify_comparacion.json"
ruta.write_text(json.dumps(log, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(log, indent=2, ensure_ascii=False)[:1500])

try:
    from google.colab import files
    files.download(str(ruta))
except ImportError:
    print("\nFuera de Colab: archivo en", ruta.resolve())

{
  "fecha_descarga": "2026-10-03T23:21:52+00:00",
  "consulta": {
    "funcion": "ox.graph_from_polygon(simplify=False) + ox.simplify_graph",
    "lugares": [
      "Miraflores, Lima, Peru",
      "San Juan de Miraflores, Lima, Peru"
    ],
    "network_type": "walk",
    "retain_all": true,
    "buffer_m": 300
  },
  "versiones": {
    "python": "3.13.15",
    "osmnx": "2.1.1",
    "networkx": "3.6.1",
    "geopandas": "1.1.4",
    "pandas": "2.2.3",
    "numpy": "2.1.3"
  },
  "sin_simplificar": {
    "nodos": 42545,
    "aristas": 102158,
    "longitud media de arista (m)": 24.552997028028706,
    "longitud total (km, aristas dirigidas / 2)": 1254.1425351946782,
    "nodos de geometría (street_count = 2)": 24711,
    "nodos de intersección (street_count >= 3)": 15729,
    "calles sin salida (street_count = 1)": 2105,
    "componentes (débiles)": 138
  },
  "simplificado": {
    "nodos": 17787,
    "aristas": 52642,
    "longitud media de arista (m)": 47.49998986092818,
    "longitu

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>